# C6 example 4/4: `SphericalKernelTensorProduct` with internal spherical harmonics

This notebook uses the complete seven-dimensional e3nn degree-3 spherical-harmonic space restricted to planar C6 and expressed in a finite-irrep basis. Each node vector generates one seven-component filter and carries invariant identity $q_a=a\in\{1,2,3\}$. The identity-weighted filters are summed first, then one spherical kernel tensor product is evaluated per layer. C6 acts around z, so every 3D vector is $(x,y)\in E_1$ plus $z\in A$.

`SphericalKernelTensorProduct.forward_from_points` accepts any covariant 3D vector at which to evaluate its owned harmonic filter. Here those points are the three node vectors themselves; there is no additional geometry argument:

$$Y_{agg}=\sum_{a=1}^{3}q_aY_{l=3}(v_a),\qquad m=\operatorname{SphericalKernelTP}\left(x,Y_{agg},w(I)\right).$$

A single degree $l=3$ contains real azimuthal modes $m=0,1,2,3$, covering every C6 frequency through $L_{full}=3$. Its restriction is $\rho_0\oplus\rho_1\oplus\rho_2\oplus\rho_3\oplus\rho_3$: seven components, including both independent Nyquist functions. `finite_irreps` is only a change of basis and discards nothing.

In [ ]:
import torch
from we3nn import CyclicGroup, RestrictedSphericalHarmonics, nn

torch.manual_seed(7)
torch.set_printoptions(precision=5, sci_mode=False)
G = CyclicGroup(6)
A = G.trivial_representation
E1 = G.standard_representation
regular = G.regular_representation()
input_rep = 3 * E1 + 5 * A
hidden_rep = 2 * regular
output_rep = E1 + 4 * A
spherical = RestrictedSphericalHarmonics(
    G, degrees=3, normalization='component', basis='finite_irreps'
)
print('spherical degrees:', spherical.degrees)
print('restricted filter representation:', spherical.rep_out.name)
print('dimensions:', input_rep.size, 'x', spherical.rep_out.size, '->', hidden_rep.size, '->', output_rep.size)

In [ ]:
def pack_input(vectors, scalars):
    xy = vectors[..., :, :2].reshape(*vectors.shape[:-2], 6)
    return torch.cat((xy, vectors[..., :, 2], scalars), dim=-1)

def unpack_input(x):
    xy = x[..., :6].reshape(*x.shape[:-1], 3, 2)
    return torch.cat((xy, x[..., 6:9].unsqueeze(-1)), dim=-1), x[..., 9:11]

def unpack_output(y):
    return torch.cat((y[..., :2], y[..., 2:3]), dim=-1), y[..., 3:6]

vectors = torch.tensor([[[1.0, 0.2, -0.4], [-0.3, 0.8, 1.2], [0.5, -0.7, 0.1]]])
scalars = torch.tensor([[0.6, -1.1]])
x = nn.RepresentationTensor(pack_input(vectors, scalars), input_rep)
Y = spherical(vectors)  # shape: [batch, 3 vector filters, 7 lossless components]
print('three internally evaluated filters Y_l(v_a):', Y)
filter_reps = spherical.rep_out.representations
irrep_totals = {rep.id: sum(other.id == rep.id for other in filter_reps) for rep in filter_reps}
irrep_seen, filter_component_labels = {}, []
for rep in filter_reps:
    irrep_seen[rep.id] = irrep_seen.get(rep.id, 0) + 1
    copy = f' copy {irrep_seen[rep.id]}/{irrep_totals[rep.id]}' if irrep_totals[rep.id] > 1 else ''
    filter_component_labels.extend(
        f'k={rep.id[0]}{copy}, component {component + 1}/{rep.size}'
        for component in range(rep.size)
    )
print('lossless C6 irrep-component labels:', filter_component_labels)

## Step 1: visualize the three filter-generating vectors

Each colored arrow is already part of `x` and serves as one point at which the degree-3 filter is evaluated. Its invariant scalar identity is printed below.

In [ ]:
import matplotlib.pyplot as plt

fig_input = plt.figure(figsize=(6, 5), constrained_layout=True)
ax = fig_input.add_subplot(111, projection='3d')
for index, vector in enumerate(vectors[0]):
    ax.quiver(0, 0, 0, *vector.tolist(), color=f'C{index}', linewidth=2, label=f'node feature v{index+1}')
limit = 1.15 * vectors.abs().max().item()
ax.set(xlim=(-limit, limit), ylim=(-limit, limit), zlim=(-limit, limit), xlabel='x', ylabel='y', zlabel='z', title='Each node vector constructs one shared filter')
ax.set_box_aspect((1, 1, 1)); ax.legend(fontsize=8)
print('scalar node features:', scalars[0].tolist())
print('vector identity scalars:', [1.0, 2.0, 3.0])
plt.show()

## Step 2: see the spherical harmonics on their domain

Spherical harmonics are functions on $S^2$. Degree $l=3$ contains all C6 frequencies 0, 1, 2, and 3. The two $k=3$ copies carry equivalent C6 transformation laws but are independent functions on the sphere, analogous to the two angular phases $\cos(3\phi)$ and $\sin(3\phi)$; both are retained. Each signed-lobe panel visualizes one finite-irrep component using radius $|Y_c|$, red for positive, and blue for negative. Components belonging to a 2D irrep must be interpreted together because rotations mix them.

In [ ]:
import math
import matplotlib.pyplot as plt
import matplotlib.colors as colors

polar = torch.linspace(0.0, math.pi, 25)
azimuth = torch.linspace(0.0, 2.0 * math.pi, 49)
polar_grid, azimuth_grid = torch.meshgrid(polar, azimuth, indexing='ij')
unit_sphere = torch.stack((
    torch.sin(polar_grid) * torch.cos(azimuth_grid),
    torch.sin(polar_grid) * torch.sin(azimuth_grid),
    torch.cos(polar_grid),
), dim=-1)
Y_sphere = spherical(unit_sphere).detach()

fig_sphere = plt.figure(figsize=(15, 8), constrained_layout=True)
sphere_axes = []
for component, label in enumerate(filter_component_labels):
    ax = fig_sphere.add_subplot(2, 4, component + 1, projection='3d')
    values = Y_sphere[..., component]
    magnitude = values.abs() / values.abs().max().clamp_min(1e-8)
    surface = unit_sphere * magnitude.unsqueeze(-1)
    facecolors = plt.cm.coolwarm((values / values.abs().max().clamp_min(1e-8) + 1.0) / 2.0)
    ax.plot_surface(surface[..., 0].numpy(), surface[..., 1].numpy(), surface[..., 2].numpy(), facecolors=facecolors, linewidth=0, antialiased=False, shade=False)
    ax.set(xlim=(-1, 1), ylim=(-1, 1), zlim=(-1, 1), title=label)
    ax.set_box_aspect((1, 1, 1)); ax.set_axis_off()
    sphere_axes.append(ax)
colorbar = fig_sphere.colorbar(plt.cm.ScalarMappable(norm=colors.Normalize(-1, 1), cmap='coolwarm'), ax=sphere_axes, shrink=0.45, pad=0.02)
colorbar.set_label('sign of normalized harmonic value')
fig_sphere.suptitle('Lossless restriction of l=3 to C6: all 7 harmonic components', fontsize=16)
plt.show()

## Step 3: build the internally filtered architecture

The contraction has the same Wigner--Eckart form as notebook 3,

$$z_o=\sum_p w_p(I)(C_p)_{oij}x_i\left[\sum_{a=1}^{3}q_aY_j(v_a)\right],$$

where the owned e3nn spherical evaluator is used to build the aggregate before contraction. The $C_p$ basis spans the complete finite-C6 Hom space after restriction; it is not limited to parent-O(3) coupling paths. Only one tensor product is evaluated per layer.

In [ ]:
class RestrictedHarmonicNetwork(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.harmonics = spherical
        self.input_layer = nn.SphericalKernelTensorProduct(
            input_rep, self.harmonics, hidden_rep, shared_weights=False
        )
        self.activation = nn.PointActiv(hidden_rep, torch.relu)
        self.output_layer = nn.SphericalKernelTensorProduct(
            hidden_rep, self.harmonics, output_rep, shared_weights=False
        )
        self.register_buffer('vector_ids', torch.tensor([1.0, 2.0, 3.0]), persistent=False)
        self.radial_in = torch.nn.Sequential(
            torch.nn.Linear(2, 16), torch.nn.SiLU(),
            torch.nn.Linear(16, self.input_layer.weight_numel),
        )
        self.radial_out = torch.nn.Sequential(
            torch.nn.Linear(2, 16), torch.nn.SiLU(),
            torch.nn.Linear(16, self.output_layer.weight_numel),
        )

    def forward(self, features):
        node_vectors, _ = unpack_input(features.tensor)
        individual_values = self.input_layer.evaluate_filter(node_vectors)
        individual_filters = nn.RepresentationTensor(individual_values, self.input_layer.rep_filter)
        identities = self.vector_ids.to(device=individual_values.device, dtype=individual_values.dtype)
        aggregated_values = (individual_values * identities.view(1, 3, 1)).sum(dim=-2)
        aggregated_filter = nn.RepresentationTensor(aggregated_values, self.input_layer.rep_filter)
        per_vector_invariants = torch.stack(
            (torch.linalg.vector_norm(node_vectors, dim=-1), node_vectors[..., 2]), dim=-1
        )
        invariant_summary = per_vector_invariants.mean(dim=-2)
        w_in = self.radial_in(invariant_summary)
        w_out = self.radial_out(invariant_summary)
        # Calling inherited forward with the aggregate performs one TP.
        h_pre = self.input_layer(features, aggregated_filter, w_in)
        h = self.activation(h_pre)
        y = self.output_layer(h, aggregated_filter, w_out)
        return y, individual_filters, aggregated_filter, w_in, w_out, h_pre, h

model = RestrictedHarmonicNetwork().eval()
y, Y_individual, Y_aggregated, w_in, w_out, h_pre, h = model(x)
print('input/output reduced-weight counts:', model.input_layer.weight_numel, model.output_layer.weight_numel)
print('vector identity scalars:', model.vector_ids.tolist())
print('one input-layer reduced-weight vector:', w_in[0])
print('aggregated filter sum_a identity[a] * Y(v_a):', Y_aggregated.tensor)
print('hidden before PointActiv:', h_pre.tensor)
print('hidden after  PointActiv:', h.tensor)
print('physical output (vector, scalars):', unpack_output(y.tensor))
kernel_basis = nn.KernelTensorProduct.sample_kernel_basis(model.input_layer, Y_aggregated)
print('single sampled basis shape [batch, paths, out, in]:', tuple(kernel_basis.shape))

## Step 4: rotate the node and all three derived filters together

Rotating `x` rotates all three vectors stored inside it. Their individual degree-3 filters and identity-weighted aggregate transform covariantly, while the radial summary and reduced-weight vector remain fixed.

In [ ]:
errors = []
for k, element in enumerate(G.elements):
    x_k = x.transform_fibers(element)
    y_k, Y_individual_k, Y_aggregated_k, w_in_k, w_out_k, _, _ = model(x_k)
    expected_k = y.transform_fibers(element)
    in_vectors_k, in_scalars_k = unpack_input(x_k.tensor)
    error = (y_k.tensor - expected_k.tensor).abs().max().item()
    errors.append(error)
    torch.testing.assert_close(y_k.tensor, expected_k.tensor, atol=8e-5, rtol=8e-5)
    torch.testing.assert_close(w_in_k, w_in, atol=1e-6, rtol=1e-6)
    torch.testing.assert_close(w_out_k, w_out, atol=1e-6, rtol=1e-6)
    out_vector_k, out_scalars_k = unpack_output(y_k.tensor)
    print(f'rotation {k}: angle={60*k:3d} degrees')
    print('  three spherical filters:', Y_individual_k.tensor[0].tolist())
    print('  identity-weighted aggregate:', Y_aggregated_k.tensor[0].tolist())
    print('  input vectors   :', in_vectors_k[0].tolist())
    print('  input scalars   :', in_scalars_k[0].tolist())
    print('  output vector   :', out_vector_k[0].tolist())
    print('  output scalars  :', out_scalars_k[0].tolist())
    print(f'  max equivariance error: {error:.3e}')

print('maximum over all rotations:', max(errors))

## Step 5: follow the spherical filter into the regular hidden state

The left heatmap shows the three complete seven-component C6 filters and $Y_{agg}=Y(v_1)+2Y(v_2)+3Y(v_3)$ as a fourth block. The right panel is the activated output of the single spherical kernel tensor product.

In [ ]:
import matplotlib.pyplot as plt

hidden_by_rotation, harmonic_by_rotation, aggregate_by_rotation = [], [], []
output_vectors, output_scalars = [], []
for element in G.elements:
    x_k = x.transform_fibers(element)
    y_k, Y_individual_k, Y_aggregated_k, _, _, _, h_k = model(x_k)
    vector_k, scalars_k = unpack_output(y_k.tensor)
    hidden_by_rotation.append(h_k.tensor[0].detach())
    harmonic_by_rotation.append(Y_individual_k.tensor[0].detach())
    aggregate_by_rotation.append(Y_aggregated_k.tensor[0].detach())
    output_vectors.append(vector_k[0].detach())
    output_scalars.append(scalars_k[0].detach())
hidden_by_rotation = torch.stack(hidden_by_rotation).cpu()
harmonic_by_rotation = torch.stack(harmonic_by_rotation).cpu()
aggregate_by_rotation = torch.stack(aggregate_by_rotation).cpu()
output_vectors = torch.stack(output_vectors).cpu()
output_scalars = torch.stack(output_scalars).cpu()
angles_deg = torch.arange(6) * 60
colors = plt.cm.hsv(torch.linspace(0, 5/6, 6).numpy())
block_size = spherical.rep_out.size

all_filter_blocks = torch.cat((harmonic_by_rotation, aggregate_by_rotation.unsqueeze(-2)), dim=-2)
flat_harmonics = all_filter_blocks.reshape(6, -1)
flat_labels = [f'{name}:{label}' for name in ('v1', 'v2', 'v3', 'agg') for label in filter_component_labels]
fig_state, (ax_harm, ax_hidden) = plt.subplots(1, 2, figsize=(17, 5), constrained_layout=True)
harmonic_image = ax_harm.imshow(flat_harmonics, aspect='auto', cmap='coolwarm')
for boundary in (block_size - 0.5, 2 * block_size - 0.5, 3 * block_size - 0.5):
    ax_harm.axvline(boundary, color='white', linewidth=2.2)
ax_harm.set(xticks=range(4 * block_size), xticklabels=flat_labels, yticks=range(6), yticklabels=[f'{a}°' for a in angles_deg.tolist()], xlabel='individual filters and identity-weighted aggregate', ylabel='C6 rotation', title='Y3(v1) | Y3(v2) | Y3(v3) | Y_agg')
ax_harm.tick_params(axis='x', labelrotation=90, labelsize=6)
fig_state.colorbar(harmonic_image, ax=ax_harm, shrink=0.75)

hidden_image = ax_hidden.imshow(hidden_by_rotation, aspect='auto', cmap='coolwarm')
ax_hidden.axvline(5.5, color='white', linewidth=2)
ax_hidden.set(xticks=range(12), yticks=range(6), yticklabels=[f'{a}°' for a in angles_deg.tolist()], xlabel='regular coordinate (copies 1 | 2)', ylabel='rotation', title='Hidden 2 Reg(C6) after PointActiv')
fig_state.colorbar(hidden_image, ax=ax_hidden, shrink=0.75)
plt.show()

## Step 6: inspect the spherical kernel tensor product

Because the complete seven-component aggregate is formed before contraction, the finite-C6 kernel basis is sampled once. One reduced-weight vector contracts its path axis, and the assertion verifies $h_{pre}=K(Y_{agg})x$.

In [ ]:
basis_for_aggregate = nn.KernelTensorProduct.sample_kernel_basis(model.input_layer, Y_aggregated)[0].detach()
effective_kernel = torch.einsum('p,poi->oi', w_in[0].detach(), basis_for_aggregate).cpu()
torch.testing.assert_close(h_pre.tensor[0], effective_kernel @ x.tensor[0], atol=3e-5, rtol=3e-5)

fig_kernel, ax_kernel = plt.subplots(figsize=(7, 5), constrained_layout=True)
kernel_image = ax_kernel.imshow(effective_kernel, aspect='auto', cmap='coolwarm')
ax_kernel.set(xlabel='input coordinate', ylabel='hidden coordinate', title='One spherical kernel from Y_aggregated')
fig_kernel.colorbar(kernel_image, ax=ax_kernel, shrink=0.75)
plt.show()

## Step 7: unpack the final spherical-kernel output

The final representation is one $xy$ vector plus four trivial coordinates. The first trivial coordinate is interpreted as vector $z$; the other three are scalar output features.

In [ ]:
fig_output = plt.figure(figsize=(12, 5), constrained_layout=True)
ax_vec = fig_output.add_subplot(1, 2, 1, projection='3d')
for k, (vector, color) in enumerate(zip(output_vectors, colors)):
    ax_vec.quiver(0, 0, 0, *vector.tolist(), color=color, linewidth=2, label=f'{60*k}°')
ax_vec.set(xlabel='x', ylabel='y', zlabel='z', title='SphericalKernelTensorProduct output vector')
output_limit = max(1e-3, 1.15 * output_vectors.abs().max().item())
ax_vec.set_xlim(-output_limit, output_limit); ax_vec.set_ylim(-output_limit, output_limit); ax_vec.set_zlim(-output_limit, output_limit); ax_vec.set_box_aspect((1, 1, 1))
ax_vec.legend(ncols=2, fontsize=7)

ax_scalar = fig_output.add_subplot(1, 2, 2)
for channel in range(3):
    ax_scalar.plot(angles_deg, output_scalars[:, channel], marker='o', label=f'output scalar {channel+1}')
ax_scalar.set(xticks=angles_deg.tolist(), xlabel='C6 rotation', ylabel='value', title='Spherical-kernel output scalars')
ax_scalar.grid(alpha=0.3); ax_scalar.legend()
plt.show()